In [ ]:
!git clone https://github.com/Jaswanth-K1210/SDAM.git 2>/dev/null || (cd SDAM && git pull)
%cd SDAM
!pip install -q timm einops transformers datasets
!pip install -q -e . --no-deps
print("Setup complete")

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

In [ ]:
import sys; sys.path.insert(0, ".")
from data.cifar10_loader import load_cifar10_features
feats = load_cifar10_features(cache_path="data/cifar10_feats.pt")
print("Categories:", list(feats.keys()))
print("Feature dim:", next(iter(feats.values())).shape[1])

In [ ]:
import torch
from sdam.model import SDAM
from sdam.utils import set_all_seeds
set_all_seeds(42)
DIM = next(iter(feats.values())).shape[1]
model = SDAM(input_dim=DIM, beta=16.0, write_threshold=0.05)
model.reset_memory()
x = torch.randn(10, DIM)
model.write(x)
x_hat = model.read(x)
assert x_hat.shape == x.shape
assert model.ssl.verify_orthogonality()
assert model.mem.n_stored > 0
print("PASSED —", model.diagnostics())

In [ ]:
!python experiments/phase2_interference.py

In [ ]:
!python experiments/phase1_retrieval.py

In [ ]:
!python experiments/phase3_curriculum.py

In [ ]:
!python experiments/seed_drift.py

In [ ]:
!python experiments/negative_control.py

In [ ]:
import glob, json
for path in sorted(glob.glob("results/*.json")):
    print("=" * 60)
    print(path)
    with open(path) as f:
        data = json.load(f)
    summary = {k: v for k, v in data.items() if not isinstance(v, (list, dict))}
    print(json.dumps(summary, indent=2))

In [ ]:
import glob
from IPython.display import Image, display
for path in sorted(glob.glob("results/*.png")):
    print(path)
    display(Image(path))

In [ ]:
from google.colab import userdata
import os
try:
    token = userdata.get("GITHUB_TOKEN")
except:
    token = ""
!git config --global user.email "jaswanthkoppisetty@gmail.com"
!git config --global user.name "Jaswanth-K1210"
!cp -r results/* results_archive/ 2>/dev/null || true
!git add results_archive/ data/cifar10_feats.pt 2>/dev/null || true
!git commit -m "results: A100 full run — real CIFAR-10, 3-way Phase 2, seed drift, negative control" || echo "nothing to commit"
if token:
    os.system(f"git remote set-url origin https://{token}@github.com/Jaswanth-K1210/SDAM.git")
    os.system("git push origin main")
    print("Pushed.")
else:
    print("Add GITHUB_TOKEN to Colab Secrets (key icon, left sidebar) then re-run.")

In [ ]:
import json, os

print("\n" + "="*64)
print("FINAL VERDICT — S-DAM FULL RUN")
print("="*64)

checks = [
    ("Phase 2 (Spelke)", "results/phase2_interference.json",
     lambda r: r.get("spelke", r).get("passed", False)),
    ("Phase 2 (Spelke > Random)", "results/phase2_interference.json",
     lambda r: r.get("spelke",{}).get("cross_mean",1) < r.get("random",{}).get("cross_mean",0)),
    ("Phase 1", "results/phase1_retrieval.json",
     lambda r: r.get("passed", False)),
    ("Phase 3", "results/phase3_curriculum.json",
     lambda r: r.get("passed", False)),
    ("Seed drift", "results/seed_drift.json",
     lambda r: r.get("passed", False)),
    ("Negative control", "results/negative_control_sst2.json",
     lambda r: r.get("passed", False)),
]

all_pass = True
for name, path, check in checks:
    try:
        r = json.load(open(path))
        ok = check(r)
    except:
        ok = False
    status = "PASS" if ok else "FAIL"
    if not ok: all_pass = False
    print(f"  {status}  {name}")

print("="*64)
print(f"  OVERALL: {'READY FOR ARXIV' if all_pass else 'FIX FAILURES FIRST'}")
print("="*64)